In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.signal import butter, filtfilt

t = np.linspace(0, 10, 1000)
fs = 1.0 / (t[1] - t[0])
current_noise = np.random.normal(0.0, np.sqrt(0.1), len(t))
prev_mean = 0.0
prev_cov = 0.1

w_amp = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Amplitude:')
w_freq = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Frequency:')
w_phase = widgets.FloatSlider(value=0.0, min=0.0, max=2*np.pi, step=0.1, description='Phase:')
w_nmean = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.1, description='Noise Mean:')
w_ncov = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description='Noise Cov.:')
w_cutoff = widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='Filter Cutoff:')
w_show = widgets.Checkbox(value=True, description='Show Noise')
w_reset = widgets.Button(description='Reset')
out = widgets.Output()

def update_plot(*args):
    global current_noise, prev_mean, prev_cov
    with out:
        out.clear_output(wait=True)
        plt.close('all')
        
        if w_nmean.value != prev_mean or w_ncov.value != prev_cov:
            current_noise = np.random.normal(w_nmean.value, np.sqrt(w_ncov.value), len(t))
            prev_mean = w_nmean.value
            prev_cov = w_ncov.value
            
        y_clean = w_amp.value * np.sin(2 * np.pi * w_freq.value * t + w_phase.value)
        y_noisy = y_clean + current_noise
        
        nyq = 0.5 * fs
        normal_cutoff = w_cutoff.value / nyq
        if normal_cutoff >= 1.0: 
            normal_cutoff = 0.99
        elif normal_cutoff <= 0: 
            normal_cutoff = 0.01
            
        b, a = butter(4, normal_cutoff, btype='low', analog=False)
        y_filtered = filtfilt(b, a, y_noisy)
        
        plt.figure(figsize=(10, 5))
        plt.plot(t, y_clean, 'b-', label='Clean', linewidth=2)
        if w_show.value:
            plt.plot(t, y_noisy, 'orange', label='Noisy', alpha=0.7)
        plt.plot(t, y_filtered, 'purple', label='Filtered', linewidth=2)
        plt.ylim(-4, 4)
        plt.legend(loc='upper right')
        plt.title("Interactive Harmonic Signal")
        plt.xlabel("Time (t)")
        plt.ylabel("Amplitude")
        plt.grid(True)
        plt.show()

w_amp.observe(update_plot, 'value')
w_freq.observe(update_plot, 'value')
w_phase.observe(update_plot, 'value')
w_nmean.observe(update_plot, 'value')
w_ncov.observe(update_plot, 'value')
w_cutoff.observe(update_plot, 'value')
w_show.observe(update_plot, 'value')

def reset_vals(b):
    w_amp.value = 1.0
    w_freq.value = 1.0
    w_phase.value = 0.0
    w_nmean.value = 0.0
    w_ncov.value = 0.1
    w_cutoff.value = 2.0
    w_show.value = True
    update_plot()

w_reset.on_click(reset_vals)

controls = widgets.VBox([
    widgets.HBox([w_amp, w_freq, w_phase]),
    widgets.HBox([w_nmean, w_ncov, w_cutoff]),
    widgets.HBox([w_show, w_reset])
])

display(controls, out)
update_plot()

Output()